# Subreddit groups v8 — manual overrides on v7

Applies the user's case-by-case corrections to `subreddit_groups_v7.parquet`. Each override is listed in `MANUAL`
with the date it was decided; `group_source` becomes `manual` and `group_v7` keeps the previous label.
Output (never overwritten): `subreddit_review_output/subreddit_groups_v8.parquet` + review xlsx.

In [ ]:
# Libraries, folders, the group list, and Emily's manual overrides (subreddit: group, date decided).
import pandas as pd
import numpy as np
import os
import re
pd.set_option('display.max_columns',None)
pd.set_option('display.max_rows',None)

SAMPLE_DIR = 'sample_1000_v4_output'
OUT_DIR = 'subreddit_review_output'
GROUPS = ['profile', 'adult', 'crypto', 'politics', 'karma_farming', 'general', 'news', 'sales', 'location', 'entertainment', 'other']
MANUAL = {                                   # subreddit: (group, date decided)
    'kidsvideos': ('entertainment', '2026-10-07'),
    'Twitch': ('entertainment', '2026-10-07'),
}

# Write a file only if it does not exist yet (never overwrite); .xlsx goes to Excel, anything else to parquet.
def save_new(df, name):
    path = os.path.join(OUT_DIR, name)
    if os.path.exists(path):
        print(f'exists, not overwritten: {path}')
        return
    if name.endswith('.xlsx'):
        df.to_excel(path, index=False, freeze_panes=(1, 1))
    else:
        df.to_parquet(path, engine='pyarrow')
    print(f'wrote {path}  {df.shape}')

In [ ]:
# Start from v7, check every override names a real subreddit and group, then apply them.
g = pd.read_parquet(os.path.join(OUT_DIR, 'subreddit_groups_v7.parquet'))
missing = set(MANUAL) - set(g['subreddit'])
assert not missing, f'not in the subreddit list: {missing}'
assert {grp for grp, _ in MANUAL.values()} <= set(GROUPS)
g['group_v7'] = g['group']
for sr, (grp, when) in MANUAL.items():
    i = g.index[g['subreddit'].eq(sr)]
    g.loc[i, ['group', 'group_source', 'manual_override_date']] = [grp, 'manual', when]
# The overridden subreddits, before and after.
g.loc[g['subreddit'].isin(MANUAL), ['subreddit', 'group_v7', 'group', 'actions', 'accounts']]

In [ ]:
# Every post and comment the sample made before its first (edit-adjusted) accusation, tagged with its subreddit's group.
accounts = pd.read_parquet(os.path.join(SAMPLE_DIR, 'sample_accounts.parquet')).set_index('username_key')
acts = []
for f in ['sample_posts.parquet', 'sample_comments.parquet']:
    d = pd.read_parquet(os.path.join(SAMPLE_DIR, f), columns=['outer_key', 'created_utc', 'subreddit'])
    acts.append(d[d['created_utc'] < d['outer_key'].map(accounts['first_mention_adj_utc'])])
acts = pd.concat(acts)
acts['group'] = acts['subreddit'].map(g.set_index('subreddit')['group'])
# Per group: subreddits, actions and accounts, and the share of all pre-accusation actions.
summary = pd.DataFrame({'subreddits': g['group'].value_counts(), 'actions': acts['group'].value_counts(),
                        'accounts': acts.groupby('group')['outer_key'].nunique()}).reindex(GROUPS)
summary['pct_actions'] = (100 * summary['actions'] / summary['actions'].sum()).round(1)
summary

In [ ]:
# Save the groups, and a review sheet sorted by group and use.
save_new(g, 'subreddit_groups_v8.parquet')
review = g.sort_values(['group', 'actions'], ascending=[True, False])[
    ['subreddit', 'group', 'group_v7', 'group_source', 'manual_override_date', 'confidence', 'reason', 'entertainment_confidence',
     'entertainment_reason', 'later_banned', 'private', 'test_subreddit', 'archived_nsfw_share', 'actions', 'accounts', 'title',
     'public_description']]
save_new(review, 'subreddit_groups_v8_review.xlsx')